# COS711 Assignment 2: Experimental Setup and Methodology

This notebook trains and evaluates the U-Net on the cache built by notebook 01, and records every run so the Results and Segmentation Enhancement sections can use them.

Each run writes its config, training history, best checkpoint and validation scores to its own folder in `runs/`. Runs that already have a `summary.json` are skipped, and a run that was cut off resumes from its last epoch.

**On Kaggle** (recommended: about 30 GPU-hours a week, and the data is already there)
1. *Settings:* Accelerator **GPU T4** (or P100) and **Internet on**. Internet is needed to clone the code and download the pretrained encoder, and requires a phone-verified Kaggle account.
2. *+ Add Input → Competitions:* add `filament-segmentation-2026`.
3. Use **Save Version → Save & Run All (Commit)**. It runs in the background without the browser open, and keeps `/kaggle/working/runs` as the version's output. Training stops starting new epochs after 10.5 h, because Kaggle ends sessions at 12 h.
4. To continue in a later version, *+ Add Input → Your Work → this notebook's output* before saving again. Earlier runs are copied in, so finished runs are skipped and unfinished ones resume.

**On Colab:** a GPU runtime, with the team's COS 711 folder on Drive (as a shared drive or a My Drive shortcut). Runs are saved to `runs/` in that folder.

| Code | Contents |
|---|---|
| `src/unet.py` | U-Net from scratch and U-Net with a pretrained ResNet-34 encoder, both with 5 output channels |
| `src/losses.py` | Multi-task loss: weighted BCE + soft Dice for the binary channels, MSE for the distance channel |
| `src/train.py` | AdamW, warmup + cosine LR, mixed precision, resumable checkpoints, best-by-validation-PQ selection |
| `src/evaluate.py` | Full-disk inference, baseline instance extraction, PQ against every annotation set, post-processing grid search, timing |

## 1. Setup

In [ ]:
# Clone the repo so src/ and notebook 01 are available. Change BRANCH to "main" once this branch is merged.
import os, subprocess, sys, time
T_START = time.time()
ON_KAGGLE = os.path.exists("/kaggle/input")
REPO_URL, BRANCH = "https://github.com/Sean-Mp/COS711-Solar-Filament.git", "chisom/experimental-setup"
REPO = "/kaggle/temp/repo" if ON_KAGGLE else "/content/repo"
if not os.path.exists(REPO):
    assert subprocess.run(["git", "clone", "-q", REPO_URL, REPO]).returncode == 0, (
        "git clone failed. On Kaggle, turn Internet on in the notebook settings.")
# Always switch to the latest BRANCH, even if an earlier run in this runtime cloned another branch
subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
assert os.path.exists(f"{REPO}/src/train.py"), (
    f"Branch {BRANCH!r} does not have this notebook's code (src/). "
    "Set BRANCH to the branch you opened this notebook from, e.g. 'chisom/experimental-setup'.")
sys.path.insert(0, REPO)

!pip -q install segmentation-models-pytorch

In [ ]:
# Where runs are saved, and how long this session may train for.
import shutil
from pathlib import Path

if ON_KAGGLE:
    # /kaggle/working is kept as the version's output. Runs from earlier versions, attached with
    # "+ Add Input" -> Your Work -> this notebook, are copied in so they are skipped or resumed.
    RUNS = Path("/kaggle/working/runs")
    RUNS.mkdir(parents=True, exist_ok=True)
    for cfg_file in Path("/kaggle/input").glob("**/runs/*/config.json"):
        src, dst = cfg_file.parent, RUNS / cfg_file.parent.name
        if not dst.exists() or ((src / "summary.json").exists() and not (dst / "summary.json").exists()):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print("copied earlier run:", src.name)
    MAX_HOURS = 10.5  # Kaggle ends sessions at 12 h; no new epoch starts after this
else:
    # The team folder holds the data zip and runs/. Members of the COS 711 shared drive see it under
    # Shareddrives; anyone it was shared with as a folder can add a My Drive shortcut to it instead
    # (Shared with me -> Organize -> Add shortcut). Notebook 01 looks in the same places.
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP_NAME = "filament-segmentation-2026.zip"
    TEAM_DIRS = [Path("/content/drive/Shareddrives/COS 711/Assignment 2"),
                 Path("/content/drive/MyDrive/COS 711/Assignment 2"),
                 Path("/content/drive/MyDrive/Assignment 2")]
    TEAM_DIR = next((d for d in TEAM_DIRS if (d / ZIP_NAME).exists()), None)
    if TEAM_DIR is None:
        raise FileNotFoundError(
            f"{ZIP_NAME} not found in any of {[str(d) for d in TEAM_DIRS]}, so notebook 01 cannot unzip the data.\n"
            "If the COS 711 folder is under 'Shared with me' in Google Drive, add a shortcut to it in My Drive, "
            "then Runtime -> Disconnect and delete runtime, and run again.")
    RUNS = TEAM_DIR / "runs"
    RUNS.mkdir(parents=True, exist_ok=True)
    MAX_HOURS = None
print("runs:", RUNS, "| completed runs:", len(list(RUNS.glob("*/summary.json"))))

In [ ]:
# Notebook 01 (data preparation) finds or unzips the data and builds the 2048x2048 cache.
# Running it here makes its objects available in this notebook: obs_meta, set_meta, FilamentDS,
# panoptic_quality, split_merge, fit_disk, normalise, circle_mask, DISK_MARGIN, CACHE, DATA.
# Its output is captured. On a fresh session this takes a while because the cache is rebuilt.
from IPython.utils.capture import capture_output

if "set_meta" not in globals():
    try:
        with capture_output() as prep_log:
            get_ipython().run_line_magic("run", f"{REPO}/notebooks/01_data_preprocessing.ipynb")
    except Exception:
        prep_log.show()
        raise
print(f"cache: {CACHE}, {shutil.disk_usage(CACHE).free / 2**30:.0f} GB still free")
print(pd.crosstab(set_meta.inst, set_meta.split, margins=True))

In [ ]:
import json, time
import matplotlib.pyplot as plt
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

from src.unet import TARGET_CHANNELS, build_model, model_size
from src.losses import MultiTaskLoss
from src.train import fit, seed_everything
from src import evaluate as ev

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
CACHE_DIR = CACHE / "scale1.0"
DEADLINE = T_START + MAX_HOURS * 3600 if MAX_HOURS else None

VAL = ev.val_items(set_meta)
print(DEVICE, "|", len(VAL), "validation observations,", sum(len(v["image_ids"]) for v in VAL), "annotation sets")

## 2. Problem formulation

Kaggle scores submissions with **Panoptic Quality**, an instance-level metric. A predicted filament counts only if it matches a ground-truth filament with IoU > 0.5, and PQ = SQ (mean IoU of matches) x RQ (an F1 score over instances). Getting the pixels right is not enough: splitting one filament into pieces or merging two neighbours costs as much as missing one.

We treat this as **bottom-up instance segmentation**. A U-Net predicts five dense maps, the targets built in notebook 01, and instances are then grouped from those maps:

| Channel | Target | Loss | Purpose |
|---|---|---|---|
| `fg` | filament mask of one annotator (optionally mixed with the consensus of all annotators) | BCE + Dice | which pixels are filament |
| `boundary` | instance edges and touching regions | BCE (pos. weight 3) + Dice | lets neighbouring filaments be separated |
| `contact` | pixels where two instances touch | BCE (pos. weight 10) + Dice | marks where to cut merged filaments |
| `dist` | distance to the edge, normalised per instance | MSE | instance centres, used as watershed markers |
| `spine` | annotated filament spine (1 px, dilated) | BCE (pos. weight 3) + Dice | helps keep thin, elongated filaments continuous |

**Why bottom-up rather than Mask R-CNN-style detection.** Filaments are thin (median width 11 px), long (up to about 1100 px) and curved, so their bounding boxes are mostly background and overlap heavily. A fully convolutional U-Net keeps full-resolution detail through its skip connections and runs on the whole 2048x2048 disk in one pass. The Segmentation Enhancement section builds the instance grouping on the extra channels (e.g. watershed on `dist` with `boundary` and `contact` as barriers). This notebook uses a deliberately simple baseline, connected components of the thresholded `fg` channel, so that the network choices can be compared under one fixed post-processing step.

**Data handling** (from notebook 01). Training uses random 512x512 crops from full-resolution disks. 80% of crops are centred on a filament (filaments cover under 1% of the disk), with random 90° rotations and flips; filaments have no preferred orientation. Pixels off the disk get zero loss weight, and the weight ramps down near the limb, where foreshortening makes annotations unreliable. Each annotation set is a separate sample, so an observation labelled by three annotators is seen with three slightly different masks. Validation runs the whole disk at once (the network is fully convolutional), scores the prediction against **every** annotation set of that observation, averages over the sets, and then averages over observations.

## 3. Experimental configuration

`BASE` holds the default settings, and each experiment overrides only what it tests. The reason for each default:

* **Optimiser:** AdamW with weight decay 1e-4, one epoch of linear warmup, then cosine decay to 1% of the peak LR. Gradients are clipped at norm 1 and training runs in fp16 (AMP) so it fits a 16 GB T4 or P100.
* **Crop 512, batch 8:** a 512 crop spans the 95th-percentile filament length (387 px, notebook 01) with context to spare, and batch 8 fits in 16 GB of GPU memory.
* **depth 6 (five poolings):** a 16x16 bottleneck on a 512 crop gives a receptive field large enough to follow long filaments, the same as a ResNet encoder.
* **Output bias initialised to p=0.01:** under 1% of pixels are positive, so this avoids a large loss spike at the start of training.
* **Model selection:** validation every 5 epochs, keeping the checkpoint with the best **PQ**, not the lowest loss.
* **Baseline post-processing:** fg threshold 0.5, minimum area 50 px (the smallest cleaned annotation is 60 px). Both are tuned later (Section 6).

In [ ]:
BASE = dict(
    seed=0,
    # model
    model="unet", base_ch=32, depth=6,               # from-scratch U-Net
    encoder="resnet34", encoder_weights="imagenet",  # used when model="smp_unet"
    in_mean=0.5, in_std=0.3,                         # input standardisation inside the model
    # data
    crop=512, batch_size=8, pos_prob=0.8, num_workers=4 if ON_KAGGLE else 2,  # Kaggle has 4 CPUs, Colab 2
    # loss
    loss_weights=dict(fg=1.0, boundary=0.5, contact=0.25, dist=0.5, spine=0.5),
    pos_weight=dict(boundary=3.0, contact=10.0, spine=3.0),
    dice=True, soft_label_alpha=0.0,
    # optimisation
    epochs=20, lr=1e-3, weight_decay=1e-4, warmup_epochs=1, grad_clip=1.0, amp=True,
    # validation
    val_every=5, pp=dict(thr=0.5, min_area=50),
)
FG_ONLY = dict(fg=1.0)

In [ ]:
# Sanity check: model sizes, plus one forward/backward pass on a real training batch.
for ov in (dict(), dict(model="smp_unet", encoder_weights=None)):
    print({**BASE, **ov}["model"], {k: round(v, 2) for k, v in model_size(build_model({**BASE, **ov})).items()})

_ds = FilamentDS(set_meta[set_meta.split == "train"], obs_meta, CACHE_DIR, crop=BASE["crop"])
_b = next(iter(DataLoader(_ds, batch_size=2)))
_m = build_model(BASE)
_loss, _parts = MultiTaskLoss(BASE["loss_weights"], BASE["pos_weight"])(_m(_b["image"]), _b)
_loss.backward()
print("loss", round(_loss.item(), 4), {k: round(v.item(), 4) for k, v in _parts.items()})
del _ds, _b, _m, _loss, _parts

In [ ]:
RESULTS_CSV = RUNS / "results.csv"

def run_experiment(name, overrides):
    """Train one configuration, select by validation PQ, then log the selected checkpoint's scores.
    Returns None if the session's time budget ran out first; rerunning resumes from last.pt."""
    cfg = {**BASE, **overrides}
    run_dir = RUNS / f"{name}_s{cfg['seed']}"
    if (run_dir / "summary.json").exists():
        print("already done:", run_dir.name)
        return json.loads((run_dir / "summary.json").read_text())
    if DEADLINE and time.time() > DEADLINE:
        print("time budget used up, not starting:", run_dir.name)
        return None
    cfg_file = run_dir / "config.json"
    if cfg_file.exists() and json.loads(cfg_file.read_text()) != json.loads(json.dumps(cfg)):
        # an unfinished run with the same name but different settings: start it again from scratch
        print(f"{run_dir.name}: settings changed since it started, discarding its unfinished checkpoint")
        for f in ("last.pt", "best.pt", "history.csv"):
            (run_dir / f).unlink(missing_ok=True)
    run_dir.mkdir(parents=True, exist_ok=True)
    cfg_file.write_text(json.dumps(cfg, indent=1))
    print(f"=== {run_dir.name}: {overrides}")

    seed_everything(cfg["seed"])
    train_ds = FilamentDS(set_meta[set_meta.split == "train"], obs_meta, CACHE_DIR,
                          crop=cfg["crop"], train=True, pos_prob=cfg["pos_prob"])
    train_dl = DataLoader(train_ds, cfg["batch_size"], shuffle=True, drop_last=True, pin_memory=True,
                          num_workers=cfg["num_workers"], persistent_workers=cfg["num_workers"] > 0)
    model = build_model(cfg).to(DEVICE)
    loss_fn = MultiTaskLoss(cfg["loss_weights"], cfg["pos_weight"], cfg["dice"], cfg["soft_label_alpha"])
    val_fn = lambda m: ev.evaluate(m, VAL, CACHE_DIR, panoptic_quality, pp_kwargs=cfg["pp"])[0]

    t0 = time.time()
    hist = fit(model, train_dl, loss_fn, cfg, run_dir, val_fn, deadline=DEADLINE)
    wall_min = (time.time() - t0) / 60
    if len(hist) < cfg["epochs"]:
        print(f"{run_dir.name} stopped after epoch {len(hist)} (time budget); it resumes next session")
        return None

    model.load_state_dict(torch.load(run_dir / "best.pt", map_location=DEVICE))
    summ, per_set = ev.evaluate(model, VAL, CACHE_DIR, panoptic_quality, split_merge, pp_kwargs=cfg["pp"])
    per_set.to_csv(run_dir / "val_per_set.csv", index=False)  # per-image scores, for best/worst cases

    summary = dict(name=name, seed=cfg["seed"], overrides=json.dumps(overrides), **model_size(model),
                   epochs=cfg["epochs"], best_epoch=int(hist.loc[hist.val_PQ.idxmax(), "epoch"]),
                   train_min=hist.train_s.sum() / 60, wall_min=wall_min,
                   **{f"val_{k}": v for k, v in summ.items()})
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=1))
    (run_dir / "last.pt").unlink()  # only needed to resume; best.pt is kept
    all_runs().to_csv(RESULTS_CSV, index=False)
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in summary.items()})
    return summary


def load_run(name, seed=0):
    "The best checkpoint of a finished run, on DEVICE in eval mode."
    run_dir = RUNS / f"{name}_s{seed}"
    cfg = json.loads((run_dir / "config.json").read_text())
    model = build_model({**cfg, "encoder_weights": None}).to(DEVICE)
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location=DEVICE))
    return model.eval()


def all_runs():
    "One row per finished run, read from the runs' summary.json files."
    return pd.DataFrame([json.loads(p.read_text()) for p in sorted(RUNS.glob("*/summary.json"))])


def results(*prefixes):
    df = all_runs()
    if df.empty:
        return df
    if prefixes:
        df = df[df.name.str.startswith(prefixes)]
    cols = ["name", "seed", "val_PQ", "val_PQ_std", "val_PQ_pooled", "val_SQ", "val_RQ", "val_dice",
            "val_one_to_many", "val_many_to_one", "best_epoch", "params_M", "train_min"]
    return df[cols].sort_values("val_PQ", ascending=False).round(4)


def best_overrides(*prefixes):
    "Overrides of the best run (by validation PQ) among runs whose names start with the prefixes."
    df = all_runs()
    df = df[df.name.str.startswith(prefixes)] if not df.empty else df
    if df.empty:
        print("no finished runs for", prefixes, "yet")
        return {}
    best = df.sort_values("val_PQ", ascending=False).iloc[0]
    print("best:", best["name"], "val PQ", round(best.val_PQ, 4))
    return json.loads(best.overrides)


def plot_history(*names, seed=0):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for n in names:
        if not (RUNS / f"{n}_s{seed}" / "history.csv").exists():
            continue
        h = pd.read_csv(RUNS / f"{n}_s{seed}" / "history.csv")
        ax[0].plot(h.epoch, h.train_loss, label=n)
        v = h.dropna(subset=["val_PQ"])
        ax[1].plot(v.epoch, v.val_PQ, "o-", label=n)
    ax[0].set(xlabel="epoch", ylabel="training loss", yscale="log"); ax[1].set(xlabel="epoch", ylabel="validation PQ")
    for a in ax: a.legend(); a.grid(alpha=0.3)
    return fig

## 4. Stage A: formulation and architecture (20 epochs each)

* **A1 vs A2:** does training the extra channels (boundary, contact, dist, spine) help, even with fg-only post-processing? If it does, they act as useful auxiliary tasks, and they are needed for the instance separation in the Enhancement section anyway.
* **A2 vs A3:** U-Net from scratch vs U-Net with an ImageNet-pretrained ResNet-34 encoder. The images are single-channel H-alpha, very different from ImageNet, so transfer learning might not help. A3 uses a lower LR (3e-4), as is usual when fine-tuning a pretrained encoder.
* *Optional A4:* the ResNet-34 U-Net without pretrained weights, to separate the effect of the architecture from the effect of pretraining.

In [ ]:
A = {
    "A1_unet_fg": dict(loss_weights=FG_ONLY),
    "A2_unet_mt": dict(),
    "A3_r34_mt":  dict(model="smp_unet", lr=3e-4),
    # "A4_r34scratch_mt": dict(model="smp_unet", encoder_weights=None),
}
for name, ov in A.items():
    run_experiment(name, ov)
results("A")

In [ ]:
plot_history(*A);

## 5. Stage B: hyperparameter tuning (on the Stage A winner)

* **B1, learning rate:** {1e-4, 3e-4, 1e-3}. The Stage A winner also competes, so its LR is not trained twice.
* **B2, using the best LR:**
  * *Soft labels:* mix the annotator's fg mask 50/50 with the consensus of all annotators of that observation. Annotators disagree on filament extent, and soft targets reduce the label noise this causes.
  * *Larger crops:* 768x768 (batch 4) for more context on long filaments.

In [ ]:
best_A = best_overrides("A")
B1 = {f"B1_lr{lr:g}": {**best_A, "lr": lr} for lr in (1e-4, 3e-4, 1e-3)
      if lr != {**BASE, **best_A}["lr"]}
for name, ov in B1.items():
    run_experiment(name, ov)
results("A", "B1")

In [ ]:
best_B1 = best_overrides("A", "B1")
B2 = {
    "B2_soft0.5": {**best_B1, "soft_label_alpha": 0.5},
    "B2_crop768": {**best_B1, "crop": 768, "batch_size": 4},
}
for name, ov in B2.items():
    run_experiment(name, ov)
results("A", "B")

## 6. Stage C: final configuration, longer training, 3 seeds

The best configuration from Stages A and B is trained for 50 epochs with three seeds, and validation PQ is reported as **mean ± std over seeds**.

The whole notebook was run twice on Kaggle (rounds 1 and 2), which measures run-to-run noise directly: the same 20-epoch configuration scores about ±0.007 differently between runs, so single-run differences under about 0.02 are not meaningful. Over both rounds, **C_final scores 0.377 ± 0.006 validation PQ (6 runs)**.

In [ ]:
FINAL = {**best_overrides("A", "B"), "epochs": 50}
for seed in (0, 1, 2):
    run_experiment("C_final", {**FINAL, "seed": seed})

final_C = results("C_final")
print(f"C_final: {len(final_C)} of 3 seeds, validation PQ {final_C.val_PQ.mean():.4f} ± {final_C.val_PQ.std():.4f}")
final_C

## 7. Stage D: training changes aimed at the errors Stage C makes

Analysing the Stage C runs (`val_per_set.csv`, `history.csv`) showed where PQ is lost:
* **False positives are the largest error.** The best seed predicts 1671 instances against 1277 ground-truth filaments: 865 true positives, **806 false positives** and 412 misses. Removing every false positive would raise RQ from 0.59 to 0.81, while finding every miss would only raise it to 0.76.
* **Context helped most.** Going from 512 to 768 crops was the largest single gain (+0.028), mostly by removing false positives (1347 → 966).
* **Two helpful changes were never combined.** Soft labels (+0.011) were tested at 512 crops only.
* **Training had not converged.** Validation PQ rose from about 0.36 to 0.38 between epochs 30 and 50, and the training loss was still falling.

Each run below changes one thing from the Stage B winner (ResNet-34 U-Net, 5 outputs, lr 3e-4, 768 crops), for 20 epochs, so it compares directly with `B2_crop768` (0.358):
* **D1, soft labels + 768 crops:** combines the two changes that helped separately.
* **D2, 1024 crops (batch 2):** more context. Batch 2 keeps the pixels per batch about the same as 512 × 8 and 768 × 4.
* **D3, 50% filament-centred crops** (instead of 80%): shows the network more filament-free regions, which targets the false positives.

**Result (round 2, 20 epochs; `B2_crop768` scored 0.3525 in this round and 0.3578 in round 1):** D1 0.3695, D2 0.3698, D3 0.3621. D3 did not reduce false positives and missed more filaments. D1 and D2 are each about +0.015 over 768 crops, borderline for single runs.

**D_final** took the best of these (D2, 1024 crops) and trained it for 80 epochs. Seed 0 scored **0.383**, only +0.007 over C_final's mean (within its ±0.006 spread, and below round 1's best C seed at 0.389), while taking **2.9× the training time** (166 vs 58 min). Training-side changes had plateaued around 0.37–0.38, so we stopped after one seed and kept **C_final** as the final model. The remaining errors (false positives and fragmented filaments) are addressed by post-processing in the Segmentation Enhancement section.

In [ ]:
best_AB = best_overrides("A", "B")
D = {
    "D1_soft_crop768": {**best_AB, "soft_label_alpha": 0.5},
    "D2_crop1024":     {**best_AB, "crop": 1024, "batch_size": 2},
    "D3_pos0.5":       {**best_AB, "pos_prob": 0.5},
}
for name, ov in D.items():
    run_experiment(name, ov)
results("B2_crop768", "D")

In [ ]:
# D_final as run in round 2: the best Stage D run (D2, 1024 crops) for 80 epochs. Only seed 0 was trained:
# it scored within C_final's seed spread at 2.9x the training time, so the final model stays C_final.
D_FINAL = {**best_overrides("B2_crop768", "D1", "D2", "D3"), "epochs": 80}
run_experiment("D_final", {**D_FINAL, "seed": 0})

final_D = results("D_final")
print(f"D_final: {len(final_D)} seed(s)" + (f", validation PQ {final_D.val_PQ.mean():.4f}" if len(final_D) else ""))
final_D

## 8. Final model and baseline post-processing

The final model is the configuration with all three seeds trained and the highest mean validation PQ, which is **C_final** (D_final has one seed). Its best seed is used for the rest of the notebook. The baseline post-processing (fg threshold × minimum area) is then tuned on its cached outputs. The tuned score is optimistic because the same validation set picks the checkpoint and the thresholds; the Kaggle leaderboard is the unbiased check. Instance separation and other post-processing are covered in the Segmentation Enhancement section.

In [ ]:
done = {n: r for n, r in (("C_final", final_C), ("D_final", final_D)) if len(r) == 3}
FINAL_DONE = bool(done)  # the cells below need a final configuration with all three seeds
if FINAL_DONE:
    for n, r in done.items():
        print(f"{n}: validation PQ {r.val_PQ.mean():.4f} ± {r.val_PQ.std():.4f} over 3 seeds")
    BEST_FINAL = max(done, key=lambda n: done[n].val_PQ.mean())
    final = done[BEST_FINAL]
    best_seed = int(final.iloc[0].seed)
    print("final model:", BEST_FINAL, "seed", best_seed)
else:
    print("No final configuration has all three seeds yet; the next session continues, and the cells below wait for it")

In [ ]:
# Tune the baseline post-processing on the final model's cached probabilities (saved, so reruns only read it).
if FINAL_DONE:
    model = load_run(BEST_FINAL, best_seed)
    grid_csv = RUNS / f"{BEST_FINAL}_s{best_seed}" / "pp_grid.csv"
    if grid_csv.exists():
        pp_grid = pd.read_csv(grid_csv)
    else:
        collected = ev.collect_probs(model, VAL, CACHE_DIR)
        pp_grid = ev.grid_search(collected, panoptic_quality,
                                 dict(thr=[0.3, 0.4, 0.5, 0.6, 0.7], min_area=[0, 25, 50, 100, 200]))
        del collected
        pp_grid.to_csv(grid_csv, index=False)
    PP = pp_grid.iloc[0][["thr", "min_area"]].to_dict()
    PP["min_area"] = int(PP["min_area"])
    print("tuned post-processing:", PP)
    display(pp_grid.head(10).round(4))

## 9. Cost: model size, training time and end-to-end inference

As the spec requires, inference is timed end to end: raw JPEG → disk fit and normalisation (notebook 01's functions) → network on the full 2048x2048 disk → instance map.

In [ ]:
def preprocess_raw(path):
    "Same preprocessing as notebook 01's process_obs at scale 1.0, for an unseen image."
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    H0, W0 = gray.shape
    p = fit_disk(gray)
    cx, cy, r = p if p is not None else (W0 / 2, H0 / 2, min(H0, W0) / 2)
    disk = circle_mask(H0, W0, cx, cy, r * DISK_MARGIN)
    img8, _, _ = normalise(gray, disk)
    return img8.astype(np.float32) / 255, disk

test_paths = sorted((DATA / "test/test_images").glob("*.jp*g"))
if FINAL_DONE:
    timing = ev.benchmark(model, test_paths[:22], preprocess_raw, pp_kwargs=PP)
    cost = dict(**model_size(model), train_min_per_run=final.train_min.mean(),
                infer_s_per_image=timing.loc["mean", "total_s"],
                device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
    (RUNS / f"{BEST_FINAL}_s{best_seed}" / "cost.json").write_text(json.dumps(cost, indent=1))
    print(cost)
    display(timing.round(4))

## 10. Save test-set predictions

Instance maps (uint16, 0 = background) for every test image from the final model, for the Results section to encode into `submission.csv` and to show predicted masks.

In [ ]:
if FINAL_DONE:
    pred_dir = RUNS / f"{BEST_FINAL}_s{best_seed}" / "test_pred"
    pred_dir.mkdir(exist_ok=True)
    for p in tqdm(test_paths):
        img, disk = preprocess_raw(p)
        inst = ev.instances_cc(ev.predict_probs(model, img), disk, **PP)
        np.savez_compressed(pred_dir / f"{p.stem}.npz", inst=inst.astype(np.uint16))
    print(len(test_paths), "test predictions written to", pred_dir)